# Agent 错误处理机制


In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from langchain.agents import create_agent
from rich import print as rprint

from numpy import extract

load_dotenv(override=True)
ZHIPU_API_KEY = os.getenv("ZHIPU_API_KEY")
ZHIPU_BASE_URL = os.getenv("ZHIPU_BASE_URL")

model=init_chat_model(
    model="glm-5.2",  # 模型名称
    model_provider="openai",
    api_key=ZHIPU_API_KEY,
    base_url=ZHIPU_BASE_URL,  # ZHIPU API 的基础 URL
)

# handle_errors 设置为True

In [2]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from rich import print as rprint

class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")


class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        handle_errors=True  # 默认行为
    )
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})

rprint(result)

{
    'messages': [
        HumanMessage(
            content='请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：
2026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='bd2808cd-4651-407f-b150-28abf35e3c3d'
        ),
        AIMessage(
            content='我来为您提取文本中的信息，并同时创建联系人信息和活动详情。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 152,
                    'prompt_tokens': 285,
                    'total_tokens': 437,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 89,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.2',
                'system_fingerprint': None,
                'id': '202607122118059f78e015e5b14a9d',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f567a-59e6-70e3-a5a1-aa7c261c8b59-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com'},
                    'id': 'call_-7453356022069977158',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventDetails',
                    'args': {'event_name': '公司年会', 'date': '2026-07-15'},
                    'id': 'call_-7453356022069977157',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 285,
                'output_tokens': 152,
                'total_tokens': 437,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 89}
            }
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) 
when only one is expected.\n Please fix your mistakes.',
            name='ContactInfo',
            id='50fd3767-283e-4d9c-a8ab-f4e5819ca4d2',
            tool_call_id='call_-7453356022069977158'
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) 
when only one is expected.\n Please fix your mistakes.',
            name='EventDetails',
            id='17655caa-11f8-4550-b1d0-dee2df337c30',
            tool_call_id='call_-7453356022069977157'
        ),
        AIMessage(
            content='看起来需要逐一调用，我先提取联系人信息：',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 58,
                    'prompt_tokens': 406,
                    'total_tokens': 464,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 22,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.2',
                'system_fingerprint': None,
                'id': '202607122118129602080261a54115',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f567a-74a5-7422-8be1-5c0207a66c0e-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
     

# handle_errors=false

In [3]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from rich import print as rprint

class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")


class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        handle_errors=False
    )
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})

rprint(result)

MultipleStructuredOutputsError: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) when only one is expected.

# handle_errors 设置为固定的字符串

In [4]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from rich import print as rprint

class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")


class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        handle_errors="请检查输入的数据"
    )
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})

rprint(result)

KeyboardInterrupt: 

# 设置为指定异常类型

In [5]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy, MultipleStructuredOutputsError, \
    StructuredOutputValidationError
from rich import print as rprint

class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")


class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        handle_errors=(MultipleStructuredOutputsError,StructuredOutputValidationError)
        # handle_errors=(StructuredOutputValidationError)
    )
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})

rprint(result)

{
    'messages': [
        HumanMessage(
            content='请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：
2026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='a1b85488-b6c0-421f-b3fc-baf8f020014a'
        ),
        AIMessage(
            content='我来帮您从文本中提取信息并调用相关功能：',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 145,
                    'prompt_tokens': 285,
                    'total_tokens': 430,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 83,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.2',
                'system_fingerprint': None,
                'id': '20260712213444039537b26fdf4916',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f5689-9709-7683-9593-e53b2bd56270-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com'},
                    'id': 'call_-7453391343881027183',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventDetails',
                    'args': {'event_name': '公司年会', 'date': '2026-07-15'},
                    'id': 'call_-7453391343881027182',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 285,
                'output_tokens': 145,
                'total_tokens': 430,
                'input_token_details': {'cache_read': 256},
                'output_token_details': {'reasoning': 83}
            }
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) 
when only one is expected.\n Please fix your mistakes.',
            name='ContactInfo',
            id='56c00945-2022-4548-a0d8-53a5b8f76553',
            tool_call_id='call_-7453391343881027183'
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) 
when only one is expected.\n Please fix your mistakes.',
            name='EventDetails',
            id='fd7b9573-f2ea-48b4-80aa-1ae897d4cc97',
            tool_call_id='call_-7453391343881027182'
        ),
        AIMessage(
            content='让我按顺序逐一调用：',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 52,
                    'prompt_tokens': 405,
                    'total_tokens': 457,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 20,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.2',
                'system_fingerprint': None,
                'id': '20260712213452381c475c70034988',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f5689-b8e2-7902-8366-3df27577f35a-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    

# 设置为自定义错误处理函数

In [ ]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy, MultipleStructuredOutputsError, \
    StructuredOutputValidationError
from rich import print as rprint

# 自定义错误处理的函数
def custom_error_handler(error : Exception) -> str:
    """自定义错误处理器"""

    error_str = str(error)

    print(f"捕获到的错误类型是：{type(error).__name__}")
    print(f"错误详情:{error_str}")

    if isinstance(error, MultipleStructuredOutputsError):
        return "检测到多个响应，请选择最相关的一个进行返回"
    elif isinstance(error, StructuredOutputValidationError):
        return "数据格式有误，请检查字段是否符号要求。"
    else :
        return f"Error:{error_str}"

class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")


class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        handle_errors=custom_error_handler
    )
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})

rprint(result)

捕获到的错误类型是：MultipleStructuredOutputsError
错误详情:Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) when only one is expected.
